# 第五阶段：智能指针

## 实验 5：`std::shared_ptr` —— 共享生命周期

`std::shared_ptr<T>` 允许多个 owner 独立延长同一个对象的生命周期。最后一个 strong owner 消失时，对象才会销毁。

本实验关注：

- 复制 `shared_ptr` 时对象与引用计数如何变化；
- control block 保存了哪些生命周期信息；
- 什么场景真正需要共享 ownership；
- 借用对象与复制 shared owner 的 API contract 有何区别；
- `make_shared`、从 `unique_ptr` 升级以及 raw pointer 陷阱；
- `shared_ptr` 的成本、线程安全边界和 ABI 限制。

默认原则仍然是：能明确唯一 owner 时优先 `unique_ptr`；只有多个实体确实需要独立控制生命周期时，才使用 `shared_ptr`。

In [1]:
// 本步骤：引入 shared_ptr、断言、字符串和移动语义所需的标准库。
#include <cassert>
#include <iostream>
#include <memory>
#include <string>
#include <utility>

### 1. 准备可观察的共享对象

下面的 `User` 输出构造和析构日志，并记录存活数量。复制 `shared_ptr` 不应复制 `User`；整个共享周期中只应看到一次构造和一次析构。

In [2]:
// 本步骤：定义带身份和生命周期日志的 User，观察最后一个 owner 何时消失。
class User
{
public:
    User(int id, std::string name)
        : id_(id), name_(std::move(name))
    {
        // 构造开始对象生命周期；shared_ptr 的复制不会再次执行这里。
        ++live_count_;
        std::cout << "User " << id_ << " created, live = " << live_count_ << '\n';
    }

    ~User() noexcept
    {
        // 最后一个 strong owner 消失时，对象才会执行一次析构。
        --live_count_;
        std::cout << "User " << id_ << " destroyed, live = " << live_count_ << '\n';
    }

    int id() const noexcept
    {
        // 返回普通值，不改变 shared ownership。
        return id_;
    }

    const std::string &name() const noexcept
    {
        // 返回只读借用，引用不能超过 User 的共享生命周期。
        return name_;
    }

    static int live_count() noexcept
    {
        // 返回当前存活对象数，供实验断言使用。
        return live_count_;
    }

private:
    int id_;
    std::string name_;
    inline static int live_count_ = 0;
};

### 2. 复制增加 strong reference count

`std::make_shared<User>()` 创建对象和对应的 control block。每个指向该 control block 的 `shared_ptr` 都是 strong owner：

```text
shared_ptr a ─┐
              ├──> control block ──> User
shared_ptr b ─┘    strong count = 2
```

复制只增加计数，不复制底层 `User`。局部副本析构时计数减少，但只要还有 owner，对象就继续存活。

In [3]:
// 本步骤：复制 shared_ptr，观察 strong count 的 1 -> 2 -> 1 变化。
{
    const int before = User::live_count();
    auto first = std::make_shared<User>(1, "Alice");

    // make_shared 返回第一个 strong owner。
    assert(first.use_count() == 1);
    assert(User::live_count() == before + 1);

    {
        // 复制 shared_ptr 创建第二个 owner，但不会构造第二个 User。
        std::shared_ptr<User> second = first;
        assert(first.use_count() == 2);
        assert(second.use_count() == 2);
        assert(first.get() == second.get());
        std::cout << "inside scope, strong owners = " << first.use_count() << '\n';
    }

    // second 析构后只减少计数，first 仍让 User 保持存活。
    assert(first.use_count() == 1);
    assert(User::live_count() == before + 1);
    std::cout << "after scope, strong owners = " << first.use_count() << '\n';
}

User 1 created, live = 1
inside scope, strong owners = 2
after scope, strong owners = 1
User 1 destroyed, live = 0


外层作用域结束后，最后一个 owner `first` 析构，strong count 降为零，`User` 随即销毁。`use_count()` 适合教学和诊断，但不应作为业务判断：它只是读取瞬间的快照，并发环境下在下一条语句前就可能变化。

### 3. 最后一个 owner 决定对象生命周期

任意一个 owner 都可以先 `reset()`，不会影响其他 owner 的有效性。对象的生命周期不再绑定某个固定变量，而是绑定整个共享 owner 集合。

In [4]:
// 本步骤：依次清空两个 owner，确认 User 只在 strong count 归零时析构。
{
    const int before = User::live_count();
    auto first = std::make_shared<User>(2, "Bob");
    auto second = first;

    // first 放弃 ownership 后，second 仍是有效的唯一 owner。
    first.reset();
    assert(!first);
    assert(second.use_count() == 1);
    assert(second->name() == "Bob");
    assert(User::live_count() == before + 1);

    // 清空最后一个 owner，使 strong count 归零并立即析构 User。
    second.reset();
    assert(!second);
    assert(User::live_count() == before);
}

User 2 created, live = 1
User 2 destroyed, live = 0


### 4. 共享 ownership 必须有真实需求

下面的 `UserStore` 会把参数保存为成员，因此调用结束后仍需要对象存活。调用方与 store 都可以独立决定何时结束自己的使用，这才是 shared ownership 的合理场景。

In [5]:
// 本步骤：定义会长期保留共享 owner 的组件。
class UserStore
{
public:
    void retain(std::shared_ptr<User> user)
    {
        // 按值参数先建立共享 ownership，再移动到成员避免额外计数操作。
        user_ = std::move(user);
    }

    const User *get() const noexcept
    {
        // 返回非拥有、可空借用；有效期受 UserStore 当前 ownership 约束。
        return user_.get();
    }

    void clear() noexcept
    {
        // store 主动放弃自己的 strong ownership。
        user_.reset();
    }

private:
    std::shared_ptr<User> user_;
};

In [6]:
// 本步骤：让调用方和 store 共享对象，再分别结束各自的 ownership。
{
    const int before = User::live_count();
    UserStore store;
    auto caller = std::make_shared<User>(3, "Carol");

    // retain 按值复制 shared_ptr，使 store 成为第二个独立 owner。
    store.retain(caller);
    assert(caller.use_count() == 2);
    assert(store.get() == caller.get());

    // 调用方结束 ownership 后，store 仍保证 User 存活。
    caller.reset();
    assert(store.get() != nullptr);
    assert(store.get()->name() == "Carol");
    assert(User::live_count() == before + 1);

    // store 清空最后一个 strong owner，对象才会析构。
    store.clear();
    assert(store.get() == nullptr);
    assert(User::live_count() == before);
}

User 3 created, live = 1
User 3 destroyed, live = 0


### 5. 借用对象不需要复制 `shared_ptr`

如果函数只在调用期间读取对象，应继续使用 `const User&` 或可空的 `const User*`。按值传 `shared_ptr<User>` 表示被调用方将共享 ownership，并可能保存副本；不要仅为了调用成员函数而增加引用计数。

In [7]:
// 本步骤：定义同步只读借用接口，它不参与 shared ownership。
void inspect(const User &user)
{
    // 只在调用期间读取 User，不复制 control block 的 strong owner。
    std::cout << "inspect " << user.id() << ": " << user.name() << '\n';
}

In [8]:
// 本步骤：对比引用借用与按值复制 shared owner 的计数变化。
{
    auto owner = std::make_shared<User>(4, "Dora");
    const long before = owner.use_count();

    // 解引用只提供 User 借用，strong count 保持不变。
    inspect(*owner);
    assert(owner.use_count() == before);

    {
        // 显式复制 shared_ptr 才建立新的 strong ownership。
        std::shared_ptr<User> another_owner = owner;
        assert(owner.use_count() == before + 1);
        inspect(*another_owner);
    }

    // 临时 owner 析构后，引用计数恢复。
    assert(owner.use_count() == before);
}

User 4 created, live = 1
inspect 4: Dora
inspect 4: Dora
User 4 destroyed, live = 0


常见签名可以这样读：

```cpp
void inspect(const User &user);
// 必须存在的同步只读借用，不延长生命周期

void retain(std::shared_ptr<User> user);
// 建立共享 ownership；函数可以保存副本并延长生命周期

void inspect_handle(const std::shared_ptr<User> &user);
// 借用 shared_ptr 本身；仅在需要读取 handle/deleter 时使用
```

另外，`const std::shared_ptr<User>` 只是禁止修改智能指针，并不会把 `User` 变成 const。需要共享只读对象时，应使用 `std::shared_ptr<const User>`。

### 6. 从唯一 ownership 升级为共享 ownership

对象通常应先由 `unique_ptr` 表达唯一 ownership。只有需求演变为真实共享时，才把它移动进 `shared_ptr`。转换会消费原 `unique_ptr`，并为共享生命周期建立 control block。没有对应的安全通用操作可以把一个可能有多个 owner 的 `shared_ptr` 直接降回 `unique_ptr`。

In [9]:
// 本步骤：把 unique_ptr 移动进 shared_ptr，观察 ownership 模型升级。
{
    const int before = User::live_count();
    auto unique_owner = std::make_unique<User>(5, "Eve");

    // 构造 shared_ptr 时消费唯一 ownership，并建立第一个 strong owner。
    std::shared_ptr<User> shared_owner = std::move(unique_owner);
    assert(!unique_owner);
    assert(shared_owner.use_count() == 1);
    assert(User::live_count() == before + 1);

    // 清空唯一的 shared owner，使对象正常析构。
    shared_owner.reset();
    assert(User::live_count() == before);
}

User 5 created, live = 1
User 5 destroyed, live = 0


### 7. 为什么优先 `make_shared`

默认写法是：

```cpp
auto user = std::make_shared<User>(1, "Alice");
```

实现通常可以把 `User` 与 control block 放进一次内存分配，减少分配次数并改善局部性。它也避免让 owning raw pointer 暂时出现在业务代码中。

不要从同一个 raw pointer 分别构造两个 `shared_ptr`：

```cpp
User *raw_owner = new User(6, "broken");
std::shared_ptr<User> first(raw_owner);  // control block A
std::shared_ptr<User> second(raw_owner); // control block B
// 两个计数都归零时会对同一 User 执行两次 delete：未定义行为
```

该反例只用于阅读，不能运行。一个资源必须从开始就属于同一个 ownership/control-block 系统。

### 8. 成本与线程安全边界

`shared_ptr` 通常涉及：

```text
shared_ptr
    │
    ├── managed pointer
    └── control block
          ├── strong reference count
          ├── weak reference count
          ├── deleter
          └── allocator / bookkeeping
```

复制和销毁 shared owner 需要维护引用计数，通常比移动 `unique_ptr` 更昂贵。不同 `shared_ptr` 实例共同管理一个 control block 时，引用计数操作可安全并发执行；这不代表底层 `User` 自动成为线程安全对象。多个线程并发修改 `User` 仍需要它自己的同步策略。

此外，strong owner 形成环时计数永远不会归零。下一实验将使用 `std::weak_ptr` 表达不延长生命周期的观察关系并打破循环。

### 9. C++ API 与 ABI 边界

`std::shared_ptr<User>` 可以作为 C++ 内部 API 表达共享 ownership，但不能直接暴露为 C ABI。它依赖 C++ 模板实例化、control block、构造析构、标准库实现和编译器 ABI。

```text
C++ Core
shared_ptr<User> / control block
        │
════════ C ABI Boundary ════════
        │
sdk_user_handle*
sdk_user_retain(handle)   // 仅在 ABI 明确支持共享计数时
sdk_user_release(handle)
        │
Kotlin/Native wrapper
```

更常见的 SDK 设计仍是单一 Kotlin owner 对应一个 opaque handle 和一个 `destroy`。只有确有跨组件共享生命周期需求时，才应把 retain/release 暴露为 ABI contract。

## 实验结论

`std::shared_ptr<T>` 应读成：**这是一个 strong owner；它与其他副本共同延长 `T` 的生命周期，最后一个 strong owner 消失时销毁对象。**

| 类型或操作 | ownership 语义 |
| --- | --- |
| `std::make_shared<T>()` | 创建对象、control block 和第一个 strong owner |
| 复制 `shared_ptr` | 增加一个 strong owner，延长生命周期 |
| 移动 `shared_ptr` | 转移一个 owner，不增加共享者数量 |
| `reset()` / owner 析构 | 放弃一个 strong ownership |
| `const T&` / `const T*` | 只借用对象，不延长生命周期 |
| `use_count()` | 诊断快照，不是业务同步条件 |

默认选择顺序：

```text
值语义 / 直接成员
        ↓ 无法直接持有时
unique_ptr
        ↓ 多个实体确实需要独立延长生命周期时
shared_ptr
```

不要因为 owner 不清楚就使用 `shared_ptr`。先明确 ownership 设计，再选择能准确表达它的类型。